In [1]:
# Preparación: cada subtarea corre en su carpeta, tal como la aprobó el revisor.
import os
from pathlib import Path
from IPython.display import Image, display

RAIZ = Path.cwd()

def mostrar_figuras(carpeta):
    for png in sorted(Path(carpeta).rglob("*.png")):
        display(Image(filename=str(png)))

## RAG mínimo en ~60 líneas (MMIA 6013 · Semana 2)

Notebook entregable: `rag_minimo_semana2.ipynb`. Implementa el pipeline completo de la sesión «S2·MIÉ — RAG mínimo en ~60 líneas»: chunking → índice → retrieval → prompt con contexto → LLM, sobre el corpus de políticas internas de NimbusSoft (`politica_vacaciones.md`, `politica_remoto.md`, `gastos.md`). Sin API, el notebook está diseñado para funcionar en «modo inspección»: muestra el prompt armado y una respuesta pre-capturada, para poder depurar el retrieval igual.

**Estado de ejecución y declaración obligatoria de lo que faltó:**

| Parte | Subtarea | Estado | Cifras medidas |
|---|---|---|---|
| Parte 1 — Chunking | T1 | Completada | Sí |
| Parte 2 — Índice (embeddings + kNN) | T2 | **Fallida tras 3 intentos** | No — sin cifras |
| Parte 3 — Generación con contexto | T3 | **Pendiente (presupuesto agotado)** | No — sin cifras |
| Diagnóstico final y cierre | T4 | Completada (redacción) | No aplica |

Longitudes del corpus medidas en T1: `politica_vacaciones.md` 784 caracteres, `politica_remoto.md` 815, `gastos.md` 824.

## Parte 1 — Chunking

In [2]:
# Subtarea T1: código aprobado (se ejecuta en trabajo/T1/)
os.chdir(RAIZ / 'trabajo' / 'T1')

# -*- coding: utf-8 -*-
"""
T1 - Parte 1 - Chunking fijo con solapamiento (Ejercicio 1.1)

Implementa chunk_fijo(texto, tamano=300, overlap=80) por caracteres, construye
las listas chunks y origen sobre el corpus DOCS con el codigo dado en el
enunciado e imprime la vista previa con el formato '[i] (origen) chunk[:70]'
seguido de tres puntos. Observa si algun chunk corta la idea de los dias no
utilizados transferibles ('hasta un maximo de 5 dias') y repite la observacion
con overlap=0, anotando las diferencias en los cortes y en el numero de chunks.

Cifras: num_chunks_overlap80, num_chunks_overlap0,
        idea_cortada_overlap80, idea_cortada_overlap0.
Esta subtarea NO produce figura PNG.
"""

import json

# Tres puntos suspensivos para la vista previa, construidos por repeticion
PUNTOS = "." * 3

# ---------------------------------------------------------------------------
# Corpus DOCS (dado en el enunciado). Vive FUERA de la solucion del ejercicio
# porque las Partes 2 y 3 lo consumen.
# ---------------------------------------------------------------------------
DOCS = {
    "politica_vacaciones.md": (
        "# Política de vacaciones\n"
        "\n"
        "## Días asignados\n"
        "\n"
        "Todo el personal tiene derecho a 23 días naturales de vacaciones\n"
        "por año natural, más los festivos que fije el calendario laboral.\n"
        "\n"
        "## Días no utilizados\n"
        "\n"
        "Los días de vacaciones no utilizados al cierre del ejercicio se\n"
        "transfieren al año siguiente, hasta un máximo de 5 días.\n"
        "\n"
        "El disfrute de los días transferidos se solicita igual que\n"
        "el ordinario y caduca el 31 de marzo del año siguiente.\n"
        "\n"
        "## Solicitud y aprobación\n"
        "\n"
        "Las vacaciones se solicitan en la herramienta de RRHH con una\n"
        "antelación mínima de 15 días naturales.\n"
        "El responsable responde en un plazo máximo de 5 días hábiles.\n"
        "\n"
        "Las bajas médicas durante las vacaciones no consumen días:\n"
        "se interrumpe el disfrute y se recupera después.\n"
        "\n"
        "Las dudas se consultan con Recursos Humanos.\n"
    ),
    "politica_remoto.md": (
        "# Política de trabajo en remoto\n"
        "\n"
        "## Modalidad híbrida\n"
        "\n"
        "El equipo trabaja en modelo híbrido: tres días en remoto y dos\n"
        "presenciales como referencia. Cada equipo fija sus días presenciales\n"
        "en la ceremonia semanal y los publica en el calendario compartido.\n"
        "\n"
        "## Equipamiento\n"
        "\n"
        "La empresa proporciona portátil, pantalla y silla ergonómica. El\n"
        "material de oficina en remoto se reembolsa según la política de\n"
        "gastos, con un límite mensual de 40 euros por persona.\n"
        "\n"
        "## Conectividad y disponibilidad\n"
        "\n"
        "En horario laboral la persona debe ser localizable y con conexión\n"
        "suficiente para videollamada. Si la conexión doméstica falla, hay\n"
        "que avisar al responsable y buscar alternativa presencial.\n"
        "\n"
        "La empresa no reembolsa la fibra doméstica, pero sí los datos\n"
        "móviles utilizados en viajes de trabajo, según la política de gastos.\n"
    ),
    "gastos.md": (
        "# Política de gastos\n"
        "\n"
        "## Regla general\n"
        "\n"
        "Todo gasto se justifica con ticket y se imputa al proyecto\n"
        "correspondiente. Las tarjetas corporativas son personales e\n"
        "intransferibles: nunca se prestan ni se comparten.\n"
        "\n"
        "## Viajes\n"
        "\n"
        "El tren en clase turista es el medio de transporte por defecto.\n"
        "El avión requiere aprobación previa del responsable cuando exista\n"
        "una alternativa en tren de menos de 4 horas de trayecto.\n"
        "\n"
        "El hotel se reserva por la plataforma corporativa, con un tope de\n"
        "120 euros por noche en capitales y 90 euros en el resto de ciudades.\n"
        "\n"
        "## Comidas y desplazamientos\n"
        "\n"
        "En viaje, la comida se cubre hasta 25 euros al día con ticket.\n"
        "Las comidas de equipo se imputan al proyecto si asisten al menos\n"
        "tres personas y se registra el motivo en la nota del gasto.\n"
        "Los taxis solo se reembolsan con justificación del horario.\n"
    ),
}

# ---------------------------------------------------------------------------
# Ejercicio 1.1 - chunking fijo con solapamiento (solucion)
# ---------------------------------------------------------------------------
def chunk_fijo(texto, tamano=300, overlap=80):
    """Divide texto en ventanas de tamano caracteres que avanzan con paso
    (tamano - overlap); el ultimo chunk puede quedar mas corto."""
    if tamano <= 0:
        raise ValueError("tamano debe ser un entero positivo")
    if not (0 <= overlap < tamano):
        raise ValueError("overlap debe cumplir 0 <= overlap < tamano")
    paso = tamano - overlap
    return [texto[i:i + tamano] for i in range(0, len(texto), paso)]


# ---------------------------------------------------------------------------
# Parametros del ejercicio e idea a vigilar
# ---------------------------------------------------------------------------
TAMANO = 300
OVERLAP_DEFECTO = 80
OVERLAP_ALT = 0
FRASE_IDEA = "hasta un máximo de 5 días"   # idea: dias no utilizados transferibles
DOC_IDEA = "politica_vacaciones.md"


def construir_chunks(docs, overlap):
    """Codigo dado en el enunciado, parametrizado con el overlap a probar."""
    chunks, origen = [], []
    for nombre, texto in docs.items():
        for c in chunk_fijo(texto, tamano=TAMANO, overlap=overlap):
            chunks.append(c)
            origen.append(nombre)
    return chunks, origen


def analizar_config(docs, overlap):
    """Fragmenta con (TAMANO, overlap) y analiza el corte de la frase idea."""
    paso = TAMANO - overlap
    chunks, origen = construir_chunks(docs, overlap)

    # chunks por documento y comprobacion con la formula teorica techo(L/paso)
    num_por_doc = {}
    for nombre, texto in docs.items():
        cs = chunk_fijo(texto, tamano=TAMANO, overlap=overlap)
        num_por_doc[nombre] = len(cs)
        esperado = -(-len(texto) // paso)
        assert len(cs) == esperado, f"desajuste con la formula en {nombre}"

    # offset de cada documento dentro de la lista global chunks
    offsets, acum = {}, 0
    for nombre in docs:
        offsets[nombre] = acum
        acum += num_por_doc[nombre]

    # localizacion de la frase idea dentro de su documento
    texto_idea = docs[DOC_IDEA]
    pos = texto_idea.find(FRASE_IDEA)
    assert pos >= 0, "la frase idea debe estar en el corpus"
    fin = pos + len(FRASE_IDEA)

    chunks_doc = chunk_fijo(texto_idea, tamano=TAMANO, overlap=overlap)

    # chunks que contienen la frase COMPLETA
    con_completa = [i for i, c in enumerate(chunks_doc) if FRASE_IDEA in c]
    assert any(FRASE_IDEA in c for c in chunks) == bool(con_completa)

    # chunks que TOCAN la frase sin contenerla entera (la cortan)
    que_cortan = []
    k = 0
    while k * paso < len(texto_idea):
        s = k * paso
        e = min(k * paso + TAMANO, len(texto_idea))
        toca = (s < fin) and (e > pos)
        entera = (s <= pos) and (fin <= e)
        if toca and not entera:
            que_cortan.append(k)
        k += 1

    extracto = None
    if que_cortan:
        k0 = que_cortan[0]
        extracto = {
            "chunk_que_corta": int(k0),
            "frontera_en_caracter": int(min(k0 * paso + TAMANO, len(texto_idea))),
            "final_del_chunk_que_corta": chunks_doc[k0][-45:],
            "inicio_del_chunk_siguiente": (chunks_doc[k0 + 1][:45]
                                           if k0 + 1 < len(chunks_doc) else ""),
        }

    return {
        "overlap": int(overlap),
        "paso": int(paso),
        "num_chunks_total": int(len(chunks)),
        "num_chunks_por_doc": {n: int(v) for n, v in num_por_doc.items()},
        "posicion_frase_en_doc": int(pos),
        "fin_frase_en_doc": int(fin),
        "idea_aparece_completa_en_algun_chunk": bool(con_completa),
        # Criterio: la idea queda CORTADA (mutilada) cuando NINGUN chunk
        # contiene la frase completa, es decir, es irrecuperable del indice.
        "idea_cortada": not con_completa,
        "chunks_con_idea_completa_doc": [int(i) for i in con_completa],
        "chunks_con_idea_completa_global": [int(offsets[DOC_IDEA] + i) for i in con_completa],
        "chunks_que_cortan_la_idea_doc": [int(i) for i in que_cortan],
        "extracto_corte": extracto,
    }


def imprimir_observacion(res):
    overlap = res["overlap"]
    print()
    print(f"--- Observacion de los cortes con overlap={overlap} "
          f"(tamano={TAMANO}, paso={res['paso']}) ---")
    print(f"Frase idea: '{FRASE_IDEA}' en {DOC_IDEA}, "
          f"caracteres {res['posicion_frase_en_doc']} a {res['fin_frase_en_doc'] - 1}")
    print(f"Chunks totales (3 documentos): {res['num_chunks_total']} | "
          f"por documento: {res['num_chunks_por_doc']}")
    if res["chunks_que_cortan_la_idea_doc"]:
        ex = res["extracto_corte"]
        print(f"  Chunk(s) que CORTAN la frase: "
              f"{res['chunks_que_cortan_la_idea_doc']} de {DOC_IDEA} "
              f"(frontera en el caracter {ex['frontera_en_caracter']})")
        print(f"    final del chunk que corta:  {ex['final_del_chunk_que_corta']!r}")
        print(f"    inicio del chunk siguiente: {ex['inicio_del_chunk_siguiente']!r}")
    else:
        print("  Ningun chunk corta la frase: no toca ninguna frontera.")
    if res["idea_aparece_completa_en_algun_chunk"]:
        print(f"  La frase SI aparece COMPLETA en el/los chunk(s) "
              f"{res['chunks_con_idea_completa_doc']} del documento "
              f"(indice global {res['chunks_con_idea_completa_global']}): "
              f"el solapamiento salva la idea.")
    else:
        print("  La frase NO aparece completa en NINGUN chunk: la idea queda "
              "mutilada e irrecuperable desde el indice.")
    print(f"  => idea_cortada (mutilada, sin recuperacion) = {res['idea_cortada']}")


# ---------------------------------------------------------------------------
# (a) overlap = 80 (por defecto): listas chunks y origen con el codigo dado
# ---------------------------------------------------------------------------
chunks, origen = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto):
        chunks.append(c)
        origen.append(nombre)

print("=" * 78)
print("VISTA PREVIA - chunk_fijo por defecto (tamano=300, overlap=80)")
print("=" * 78)
for i, c in enumerate(chunks):
    print(f"[{i}] ({origen[i]}) {c[:70]}{PUNTOS}")

res80 = analizar_config(DOCS, OVERLAP_DEFECTO)
imprimir_observacion(res80)

# ---------------------------------------------------------------------------
# (b) overlap = 0: se repite la observacion
# ---------------------------------------------------------------------------
chunks0, origen0 = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto, tamano=TAMANO, overlap=OVERLAP_ALT):
        chunks0.append(c)
        origen0.append(nombre)

print()
print("=" * 78)
print("VISTA PREVIA - chunk_fijo con overlap=0 (tamano=300, overlap=0)")
print("=" * 78)
for i, c in enumerate(chunks0):
    print(f"[{i}] ({origen0[i]}) {c[:70]}{PUNTOS}")

res0 = analizar_config(DOCS, OVERLAP_ALT)
imprimir_observacion(res0)

# ---------------------------------------------------------------------------
# Comparacion overlap=80 frente a overlap=0
# ---------------------------------------------------------------------------
num80 = res80["num_chunks_total"]
num0 = res0["num_chunks_total"]
inflacion_emp = num80 / num0
inflacion_teo = TAMANO / (TAMANO - OVERLAP_DEFECTO)

print()
print("=" * 78)
print("COMPARACION overlap=80 frente a overlap=0")
print("=" * 78)
print(f"num_chunks_overlap80 = {num80}  {res80['num_chunks_por_doc']}")
print(f"num_chunks_overlap0  = {num0}  {res0['num_chunks_por_doc']}")
print(f"Inflacion empirica del indice num80/num0 = {inflacion_emp:.6f} "
      f"(mas {(inflacion_emp - 1) * 100:.2f} por ciento)")
print(f"Inflacion teorica C/(C-V) = {TAMANO}/{TAMANO - OVERLAP_DEFECTO} = "
      f"{inflacion_teo:.6f} (mas {(inflacion_teo - 1) * 100:.2f} por ciento)")
print(f"idea_cortada_overlap80 = {res80['idea_cortada']} "
      f"(frase completa en algun chunk: {res80['idea_aparece_completa_en_algun_chunk']})")
print(f"idea_cortada_overlap0  = {res0['idea_cortada']} "
      f"(frase completa en algun chunk: {res0['idea_aparece_completa_en_algun_chunk']})")

ex80 = res80["extracto_corte"]
ex0 = res0["extracto_corte"]
notas = (
    f"Con overlap=80 (paso {res80['paso']}) la frase idea '{FRASE_IDEA}' "
    f"(caracteres {res80['posicion_frase_en_doc']} a {res80['fin_frase_en_doc'] - 1} "
    f"de {DOC_IDEA}) queda cortada en la frontera del caracter "
    f"{ex80['frontera_en_caracter']}: el chunk {ex80['chunk_que_corta']} termina con "
    f"{ex80['final_del_chunk_que_corta']!r}, pero la ventana solapada siguiente la "
    f"contiene COMPLETA (chunk {res80['chunks_con_idea_completa_doc']} del documento), "
    f"de modo que la idea es recuperable: idea_cortada_overlap80=False. "
    f"Con overlap=0 (paso {res0['paso']}) la misma frase cae sobre la frontera del "
    f"caracter {ex0['frontera_en_caracter']}: el chunk {ex0['chunk_que_corta']} termina "
    f"con {ex0['final_del_chunk_que_corta']!r} y el chunk siguiente empieza con "
    f"{ex0['inicio_del_chunk_siguiente']!r}; ningun chunk la contiene completa y la "
    f"idea queda mutilada en los dos fragmentos: idea_cortada_overlap0=True. "
    f"Diferencia en los cortes: en ambos casos algun chunk corta la frase, pero con "
    f"solapamiento el corte se repara en la ventana siguiente, mientras que sin "
    f"solapamiento la frontera parte la frase en dos mitades irrecuperables. "
    f"Coste del solapamiento: el indice pasa de {num0} a {num80} chunks (factor "
    f"{inflacion_emp:.4f}); la formula teorica C/(C-V) = "
    f"{TAMANO}/{TAMANO - OVERLAP_DEFECTO} da un factor {inflacion_teo:.4f} "
    f"(mas {(inflacion_teo - 1) * 100:.1f} por ciento)."
)
print()
print("NOTA DE OBSERVACION:")
print(notas)

# ---------------------------------------------------------------------------
# Guardado de todas las cifras en resultados.json
# ---------------------------------------------------------------------------
resultados = {
    "tarea": "T1 - Parte 1: chunking fijo con solapamiento (Ejercicio 1.1)",
    "parametros": {
        "tamano_chunk_caracteres": int(TAMANO),
        "overlap_por_defecto": int(OVERLAP_DEFECTO),
        "overlap_alternativo": int(OVERLAP_ALT),
    },
    "corpus": {
        "documentos": list(DOCS.keys()),
        "longitud_docs_caracteres": {n: int(len(t)) for n, t in DOCS.items()},
    },
    "frase_idea": FRASE_IDEA,
    "doc_de_la_idea": DOC_IDEA,
    "criterio_idea_cortada": ("True si NINGUN chunk contiene la frase completa "
                              "(idea mutilada e irrecuperable desde el indice)"),
    "detalle_overlap80": res80,
    "detalle_overlap0": res0,
    # ---- cifras pedidas ----
    "num_chunks_overlap80": int(num80),
    "num_chunks_overlap0": int(num0),
    "idea_cortada_overlap80": bool(res80["idea_cortada"]),
    "idea_cortada_overlap0": bool(res0["idea_cortada"]),
    # ---- lecturas complementarias del mismo analisis ----
    "idea_aparece_completa_en_algun_chunk_overlap80": bool(res80["idea_aparece_completa_en_algun_chunk"]),
    "idea_aparece_completa_en_algun_chunk_overlap0": bool(res0["idea_aparece_completa_en_algun_chunk"]),
    "algun_chunk_corta_la_frase_overlap80": bool(res80["chunks_que_cortan_la_idea_doc"]),
    "algun_chunk_corta_la_frase_overlap0": bool(res0["chunks_que_cortan_la_idea_doc"]),
    "inflacion_indice_empirica_num80_entre_num0": float(inflacion_emp),
    "inflacion_indice_teorica_C_sobre_C_menos_V": float(inflacion_teo),
    "inflacion_porcentual_teorica": float((inflacion_teo - 1) * 100),
    "notas_observacion": notas,
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print()
print("CIFRAS GUARDADAS EN resultados.json:")
print(f"  num_chunks_overlap80   = {resultados['num_chunks_overlap80']}")
print(f"  num_chunks_overlap0    = {resultados['num_chunks_overlap0']}")
print(f"  idea_cortada_overlap80 = {resultados['idea_cortada_overlap80']}")
print(f"  idea_cortada_overlap0  = {resultados['idea_cortada_overlap0']}")

os.chdir(RAIZ)
mostrar_figuras(RAIZ / 'trabajo' / 'T1')

VISTA PREVIA - chunk_fijo por defecto (tamano=300, overlap=80)
[0] (politica_vacaciones.md) # Política de vacaciones

## Días asignados

Todo el personal tiene de...
[1] (politica_vacaciones.md) es no utilizados al cierre del ejercicio se
transfieren al año siguien...
[2] (politica_vacaciones.md)  Solicitud y aprobación

Las vacaciones se solicitan en la herramienta...
[3] (politica_vacaciones.md) vacaciones no consumen días:
se interrumpe el disfrute y se recupera d...
[4] (politica_remoto.md) # Política de trabajo en remoto

## Modalidad híbrida

El equipo traba...
[5] (politica_remoto.md) lica en el calendario compartido.

## Equipamiento

La empresa proporc...
[6] (politica_remoto.md) os por persona.

## Conectividad y disponibilidad

En horario laboral ...
[7] (politica_remoto.md) ternativa presencial.

La empresa no reembolsa la fibra doméstica, per...
[8] (gastos.md) # Política de gastos

## Regla general

Todo gasto se justifica con ti...
[9] (gastos.md) 

El tren en clase turi

### Ejercicio 1.1 — `chunk_fijo(texto, tamano=300, overlap=80)`

Implementación sobre caracteres que devuelve una lista de strings: ventanas de 300 caracteres con avance `tamano − overlap`. Con `overlap=80` el paso es 220; con `overlap=0`, 300. El corpus fragmentado se construye fuera del ejercicio (las Partes 2 y 3 lo consumen) y cada chunk registra su documento de origen.

### Resultados medidos

**Número de chunks.** Con `overlap=80` el índice queda en 12 chunks (4 por documento); con `overlap=0`, en 9 chunks (3 por documento).

**La frase idea: «hasta un máximo de 5 días»** (política de transferencia de vacaciones, `politica_vacaciones.md`, caracteres 294 a 319). La frontera del carácter 300 la corta en ambos escenarios: el chunk 0 termina en `cicio se\ntransfieren al año siguiente, hasta `.

- **Con `overlap=80`** (paso 220): aunque el chunk 0 corta la frase, la ventana solapada siguiente (`es no utilizados al cierre del ejercicio se\nt...`) la contiene completa: es el chunk [1] del documento (chunk [1] global). La idea es recuperable desde el índice: `idea_cortada_overlap80 = False`.
- **Con `overlap=0`** (paso 300): el chunk 0 termina en `...hasta ` y el chunk siguiente empieza en `un máximo de 5 días.\n\nEl disfrute de los días`. Ningún chunk contiene la frase completa: queda partida en dos mitades irrecuperables desde el índice (chunks que la cortan: [0, 1]). `idea_cortada_overlap0 = True`.

| Escenario | Paso | Chunks totales | ¿Algún chunk corta la frase? | ¿Frase completa en algún chunk? | Idea recuperable |
|---|---|---|---|---|---|
| overlap=80 | 220 | 12 | Sí (chunk 0) | Sí (chunk [1]) | Sí (`False` = no cortada) |
| overlap=0 | 300 | 9 | Sí (chunks 0 y 1) | No | No (`True` = cortada) |

**Coste del solapamiento.** El índice pasa de 9 a 12 chunks: inflación empírica 12/9 = 1.3333. La fórmula teórica C/(C−V) = 300/220 da 1.3636, es decir, un 36.3636 % más de chunks.

**Reflexión.** El resultado confirma lo que anticipaba el enunciado: con `overlap=80` la idea aparece completa en algún chunk aunque otro la corte; con `overlap=0` la misma frontera la mutila. En ambos casos «algún chunk corta la frase»; la diferencia es que el solapamiento repara el corte en la ventana siguiente, mientras que sin solapamiento la frontera parte la frase en dos mitades irrecuperables. Como dicen las diapositivas de la semana, «ninguna búsqueda repara un mal corte»: si la frase no está completa en ningún fragmento, ningún kNN la traerá entera.

## Parte 2 — Índice (embeddings + kNN)

**Estado: subtarea fallida tras 3 intentos.** No hay cifras medidas de esta parte y se declara explícitamente: no se obtuvo la forma del índice (`V.shape`), ni el motor efectivo (denso con `sentence-transformers` o fallback léxico TF-IDF), ni el conteo de tokens del fragmento más largo contra el tope del modelo, ni una validación de `recuperar`.

Lo que esta parte debía producir, según el enunciado:

1. **Índice**: embeddings normalizados de los 12 chunks de la Parte 1 con `paraphrase-multilingual-MiniLM-L12-v2` (o fallback TF-IDF normalizado), imprimiendo `Retrieval: <MOTOR> | índice: V.shape`.
2. **El techo del modelo, medido**: contar los tokens del fragmento más largo con el tokenizador del mismo modelo y compararlos con `max_seq_length`, porque el chunking de la Parte 1 cuenta caracteres y el modelo trunca en tokens «sin avisar». El contexto del curso: el techo del notebook son 128 tokens según las diapositivas, la regla del material es 500 tokens «aprox.», las notas teóricas y el Lab 02 usan 512, y bge-m3 (H200 de la USFQ) llega a 8192. Esa medición **no pudo ejecutarse** aquí.
3. **Ejercicio 2.1 — `recuperar(pregunta, k=3)`**: devuelve `[(score, idx_chunk), ...]` ordenada de mayor a menor score usando el índice `V` y `vec_consulta(pregunta)`, con esa firma exacta porque la Parte 3 y el ejercicio 3.1 la llaman así.

**Consecuencia en cadena:** sin `V` ni `recuperar` no puede armarse el prompt con contexto ni ejecutarse las pruebas de la Parte 3; por eso la subtarea T3 quedó pendiente.

## Parte 3 — Generación con contexto

**Estado: subtarea pendiente — presupuesto agotado.** Sin cifras medidas: no hay respuestas generadas, ni listados de chunks recuperados por pregunta, ni detección de abstención. Se declara explícitamente.

El diseño que quedaba por ejecutar, según el enunciado:

- **Prompt baseline**: contexto delimitado por `<contexto>...</contexto>`, instrucción de responder SOLO con base en él y permiso explícito de decir «no sé». `armar_prompt(pregunta, k=3)` inserta los chunks como `[i] {chunks[i]}` para los k=3 resultados de `recuperar`.
- **Abstención única en todo el curso**: `ABSTENCION = "El corpus no contiene información suficiente."` — la misma frase del Lab 02 (`rag_pipeline.py`) y del golden set del sábado. `se_abstuvo` la detecta comparando normalizado (minúsculas, sin tildes, sin puntuación), no por igualdad de cadena.
- **`generar` en cascada**: API de Anthropic → H200 de la USFQ (vLLM, `GLM-5.3-Flash`, temperature 0.1, `max_tokens` 1500 porque el razonamiento se paga del mismo cupo) → Ollama local (`llama3.1:8b`) → **modo inspección** con respuestas pre-capturadas (etiquetas `transferir`, `exterior`, `mascotas`, `ambigua`).

**Ejercicio 3.1 — las tres pruebas de fuego** (listadas, sin resultados medidos):

1. «¿puedo trabajar desde el exterior?» — respuesta clara en `politica_remoto.md`.
2. «¿puedo llevar a mi mascota a la oficina?» — sin respuesta en el corpus; el mejor comportamiento posible es la frase de abstención. Punto clave del enunciado: «Un RAG que siempre responde algo es un generador de alucinaciones con citas decorativas».
3. «¿qué necesita aprobación del gerente?» — ambigua entre documentos (aprobación del gerente de área en `gastos.md`).

Para cada una se debía anotar qué chunks recuperó (score, índice y origen) y si la respuesta se apoya en ellos. **No se ejecutó ninguna.**

## Diagnóstico final y cierre

**¿Cuál de las 4 cajas del pipeline (ingesta / chunking / retrieval / generación) es la más frágil y cómo vigilarla?**

Con la evidencia medida en este notebook, la caja más frágil es la **fragmentación (chunking)**:

1. **Lo demostró la Parte 1 con cifras**: la misma frontera del carácter 300 deja la frase idea recuperable con `overlap=80` (12 chunks) y mutilada de forma irrecuperable con `overlap=0` (9 chunks). El corte se decide una vez, en la ingesta, y condiciona a todas las demás cajas.
2. **El curso lo afirma**: «Cambiar el corte cambia los cuatro papeles a la vez, y solo el primero se nota» — el fragmento se embebe, se indexa con su identificador, se recupera y se cita, y se compara con el golden set — y «la fragmentación acota a la recuperación: ninguna búsqueda repara un mal corte».
3. **Además, la unidad de conteo es sospechosa**: el chunking cuenta caracteres (300) pero el modelo trunca en tokens. El notebook debía verificar el fragmento más largo contra `max_seq_length` (techo de 128 tokens del MiniLM según las diapositivas; 512 en el Lab 02; 8192 en bge-m3) y esa verificación quedó sin ejecutar por la falla de la Parte 2 — motivo adicional para no confiarse del «probablemente cabe».

**Cómo vigilarla en mi proyecto:**

- Medir antes de indexar: contar los tokens del fragmento más largo con el tokenizador del propio modelo y avisar si no cabe (la conducta del Lab 02: fragmentar en tokens —512, sin pasar del tope − 2, con solape de un quinto—).
- Pruebas de frase-idea como la del Ejercicio 1.1: verificar que las ideas críticas aparecen completas en algún chunk, con overlap y sin él, aceptando el coste de índice medido (aquí, factor 1.3333 empírico frente a 1.3636 teórico).
- Ante una respuesta mala, diagnosticar en qué caja falló (¿el fragmento correcto no llegó al top-k, o llegó y el modelo lo ignoró?), como en la actividad «El chunk culpable».
- Desde mañana, métricas sobre un golden set — Hit Rate, MRR y Recall@k —, como anuncia el cierre de la semana.

**Cierre / estado final.** El diseño del notebook cubre el pipeline completo — chunking con overlap, índice, retrieval top-k y generación con contexto delimitado y derecho a decir «no sé» —, pero en esta ejecución solo la Parte 1 quedó verificada con cifras medidas. **Lo que faltó (declaración):** T2 (Parte 2 — Índice de embeddings + kNN y función `recuperar`), fallida tras 3 intentos, y T3 (Parte 3 — Generación con contexto y las tres pruebas de fuego), pendiente por presupuesto agotado. Las cifras de retrieval y de generación quedan, por tanto, sin medir en este entregable.